In [3]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model

In [9]:
#Load the training data set from data/
X_train = pd.read_csv('data/X_train.csv')
y_train = pd.read_csv('data/y_test.csv')

In [5]:
X_train.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,SmallTransaction,LargeTransaction,RegularTransaction,Hour,Day
0,-1.128889,1.227562,-0.175061,-1.260058,-0.506687,2.050260,3.244372,-0.399434,0.766429,-0.094691,...,0.455373,0.183782,-0.042999,0.016653,-0.150805,False,False,True,-0.521763,-0.983852
1,-1.622888,-9.264831,7.281645,-3.141222,-2.603790,-3.199878,-1.151250,-1.958937,2.852666,6.048398,...,0.952161,0.682165,1.238353,0.198208,-0.354411,True,False,False,-1.720596,-0.983852
2,1.018200,2.239052,-1.468257,-0.415760,-1.651330,-1.518224,-0.328507,-1.458644,0.004632,-1.217365,...,-0.595912,-0.483846,0.034075,-0.047606,-0.292800,False,False,True,0.163285,1.016413
3,-0.644447,-0.835083,0.353545,1.163347,-0.032913,1.277157,-0.710869,0.950491,-0.064290,-0.600600,...,0.139554,-0.489869,-0.104113,-0.114724,-0.300714,False,False,True,0.505808,-0.983852
4,0.696663,-1.048851,-1.866900,0.064047,0.818932,0.888045,-0.308135,0.717697,-0.106841,0.315916,...,-1.599866,-0.762909,0.111594,0.191250,0.999370,False,True,False,-0.521763,1.016413


In [22]:
y_train.head()

,Class
0,0
1,0
2,0
3,0
4,0


In [6]:
#Get the amount of columns, this will be the size of our input layer
input_dim = X_train.shape[1] 

input_layer = layers.Input(shape=(input_dim, ))

#Compression stack for the autoencoder (35 -> 16 -> ...)
x = layers.Dense(16, activation='relu')(input_layer)
x = layers.Dense(8, activation='relu')(x)
x = layers.Dense(4, activation='relu')(x)
bottleneck = layers.Dense(2, activation='relu')(x) #The last layer before we build the decoder

#Decoder
x = layers.Dense(4, activation='relu')(bottleneck)
x = layers.Dense(8, activation='relu')(x)
x = layers.Dense(16, activation='relu')(x)
output_layer = layers.Dense(input_dim, activation='linear')(x)

In [7]:
autoencoder = Model(inputs=input_layer, outputs=output_layer, name = 'Anomaly_Detection')
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

Model: "Anomaly_Detection"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 35)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │            10 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 35)             │           595 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,549 (6.05 KB)

 Trainable params: 1,549 (6.05 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
autoencoder.fit(X_train, y_train, epochs=30)

KeyError: 1